# RA1 — Preparação dos dados de COVID-19 na Índia

Este notebook reúne a leitura, a avaliação da qualidade e a preparação dos dados.
Execute as células na ordem. Os três CSVs originais são as únicas entradas necessárias.
No Colab, envie os arquivos quando solicitado. No VS Code, abra este notebook na pasta do projeto e selecione um kernel com Pandas instalado.

## 1. Definição do problema e perguntas analíticas

Queremos compreender a evolução dos registros de COVID-19, da testagem e da vacinação entre estados e territórios da Índia. Primeiro precisamos preparar bases com períodos diferentes, lacunas e nomes territoriais inconsistentes.

- Como os registros acumulados de casos, recuperações e óbitos evoluíram em cada estado no período disponível?
- Como o volume registrado de testes variou entre estados e ao longo do tempo?
- Como a administração da primeira e da segunda dose evoluiu nos estados durante 2021?

A RA1 termina com os dados preparados. As perguntas serão respondidas nas próximas etapas.

## 2. Fontes de dados

Os arquivos foram fornecidos como provenientes do conjunto **COVID-19 in India**, de Sudalai Rajkumar, distribuído no Kaggle. A data e a versão do download não foram informadas; a origem primária específica de cada arquivo não foi confirmada.

| Arquivo | Formato | Principais campos |
|---|---|---|
| covid_19_india.csv | CSV UTF-8, separado por vírgulas | Data, território, confirmados, recuperados e óbitos |
| StatewiseTestingDetails.csv | CSV UTF-8, separado por vírgulas | Data, estado, amostras, negativos e positivos |
| covid_vaccine_statewise.csv | CSV UTF-8, separado por vírgulas | Data, estado, doses e recortes por imunizante, idade e sexo |

São dados históricos. As quantidades de registros, colunas e amostras aparecem após a leitura. Todas as variáveis serão preservadas; o dicionário em `docs/data_dictionary.md` explica seus significados.

## 3. Ingestão dos dados

Vamos ler os valores como texto para examinar o conteúdo original antes de converter datas e números. Procuramos os arquivos na pasta atual ou na pasta acima; no Colab, também é possível enviá-los pelo seletor de arquivos.

In [3]:
from pathlib import Path
import hashlib
import pandas as pd

arquivos = {
    "casos": "covid_19_india.csv",
    "testes": "StatewiseTestingDetails.csv",
    "vacinacao": "covid_vaccine_statewise.csv",
}

pasta = Path.cwd()
for candidata in [Path.cwd(), Path.cwd().parent]:
    if all((candidata / arquivo).is_file() for arquivo in arquivos.values()):
        pasta = candidata
        break

faltantes = [arquivo for arquivo in arquivos.values() if not (pasta / arquivo).is_file()]
if faltantes:
    try:
        from google.colab import files
    except ImportError:
        raise FileNotFoundError("Coloque os três CSVs na raiz do projeto. Faltam: " + ", ".join(faltantes)) from None
    print("Envie os arquivos: " + ", ".join(faltantes))
    enviados = files.upload()
    # Salvar apenas entradas que ainda não existem, sem sobrescrever originais.
    for arquivo in faltantes:
        if arquivo in enviados and not (pasta / arquivo).exists():
            (pasta / arquivo).write_bytes(enviados[arquivo])
    faltantes = [arquivo for arquivo in arquivos.values() if not (pasta / arquivo).is_file()]
    if faltantes:
        raise FileNotFoundError("Ainda faltam: " + ", ".join(faltantes))

hashes_originais = {
    nome: hashlib.sha256((pasta / arquivo).read_bytes()).hexdigest()
    for nome, arquivo in arquivos.items()
}
originais = {
    nome: pd.read_csv(pasta / arquivo, dtype="string", keep_default_na=False, encoding="utf-8-sig")
    for nome, arquivo in arquivos.items()
}
display(pd.DataFrame([
    {"base": nome, "arquivo": arquivos[nome], "registros": len(df), "colunas": len(df.columns)}
    for nome, df in originais.items()
]))
for nome, df in originais.items():
    print(nome)
    display(df.head(3))

,base,arquivo,registros,colunas
0,casos,covid_19_india.csv,18110,9
1,testes,StatewiseTestingDetails.csv,16336,5
2,vacinacao,covid_vaccine_statewise.csv,7845,24


casos


,Sno,Date,Time,State/UnionTerritory,ConfirmedIndianNational,ConfirmedForeignNational,Cured,Deaths,Confirmed
0,1,2020-01-30,6:00 PM,Kerala,1,0,0,0,1
1,2,2020-01-31,6:00 PM,Kerala,1,0,0,0,1
2,3,2020-02-01,6:00 PM,Kerala,2,0,0,0,2


testes


,Date,State,TotalSamples,Negative,Positive
0,2020-04-17,Andaman and Nicobar Islands,1403.0,1210,12.0
1,2020-04-24,Andaman and Nicobar Islands,2679.0,,27.0
2,2020-04-27,Andaman and Nicobar Islands,2848.0,,33.0


vacinacao


,Updated On,State,Total Doses Administered,Sessions,Sites,First Dose Administered,Second Dose Administered,Male (Doses Administered),Female (Doses Administered),Transgender (Doses Administered),...,18-44 Years (Doses Administered),45-60 Years (Doses Administered),60+ Years (Doses Administered),18-44 Years(Individuals Vaccinated),45-60 Years(Individuals Vaccinated),60+ Years(Individuals Vaccinated),Male(Individuals Vaccinated),Female(Individuals Vaccinated),Transgender(Individuals Vaccinated),Total Individuals Vaccinated
0,16/01/2021,India,48276.0,3455.0,2957.0,48276.0,0.0,,,,...,,,,,,,23757.0,24517.0,2.0,48276.0
1,17/01/2021,India,58604.0,8532.0,4954.0,58604.0,0.0,,,,...,,,,,,,27348.0,31252.0,4.0,58604.0
2,18/01/2021,India,99449.0,13611.0,6583.0,99449.0,0.0,,,,...,,,,,,,41361.0,58083.0,5.0,99449.0


## 4. Avaliação da qualidade dos dados

Primeiro verificamos ausências, duplicatas e conversões. Na leitura inicial todos os tipos são texto por escolha nossa; isso não significa que a fonte esteja errada. Datas e contagens serão verificadas com os formatos esperados.

Os nomes curtos abaixo correspondem às mesmas colunas originais. O mapa explícito também permite identificar uma mudança inesperada nos arquivos.

In [4]:
mapa_colunas = {
    "casos": {
        "Sno": "sno",
        "Date": "date",
        "Time": "time",
        "State/UnionTerritory": "state",
        "ConfirmedIndianNational": "confirmed_indian_national",
        "ConfirmedForeignNational": "confirmed_foreign_national",
        "Cured": "cured",
        "Deaths": "deaths",
        "Confirmed": "confirmed"
    },
    "testes": {
        "Date": "date",
        "State": "state",
        "TotalSamples": "total_samples",
        "Negative": "negative",
        "Positive": "positive"
    },
    "vacinacao": {
        "Updated On": "date",
        "State": "state",
        "Total Doses Administered": "total_doses_administered",
        "Sessions": "sessions",
        " Sites ": "sites",
        "First Dose Administered": "first_dose_administered",
        "Second Dose Administered": "second_dose_administered",
        "Male (Doses Administered)": "male_doses_administered",
        "Female (Doses Administered)": "female_doses_administered",
        "Transgender (Doses Administered)": "transgender_doses_administered",
        " Covaxin (Doses Administered)": "covaxin_doses_administered",
        "CoviShield (Doses Administered)": "covi_shield_doses_administered",
        "Sputnik V (Doses Administered)": "sputnik_v_doses_administered",
        "AEFI": "aefi",
        "18-44 Years (Doses Administered)": "18_44_years_doses_administered",
        "45-60 Years (Doses Administered)": "45_60_years_doses_administered",
        "60+ Years (Doses Administered)": "60_plus_years_doses_administered",
        "18-44 Years(Individuals Vaccinated)": "18_44_years_individuals_vaccinated",
        "45-60 Years(Individuals Vaccinated)": "45_60_years_individuals_vaccinated",
        "60+ Years(Individuals Vaccinated)": "60_plus_years_individuals_vaccinated",
        "Male(Individuals Vaccinated)": "male_individuals_vaccinated",
        "Female(Individuals Vaccinated)": "female_individuals_vaccinated",
        "Transgender(Individuals Vaccinated)": "transgender_individuals_vaccinated",
        "Total Individuals Vaccinated": "total_individuals_vaccinated"
    }
}

marcadores_ausentes = ["", "-", "NA", "N/A", "NaN", "nan", "null"]
formatos_data = {"casos": "%Y-%m-%d", "testes": "%Y-%m-%d", "vacinacao": "%d/%m/%Y"}
avaliacao = {}
ausencias_antes = {}
numericas = {}

for nome, original in originais.items():
    if list(original.columns) != list(mapa_colunas[nome]):
        raise ValueError(f"Colunas inesperadas em {nome}; confira o arquivo de entrada.")
    df = original.rename(columns=mapa_colunas[nome]).copy()
    for coluna in df:
        df[coluna] = df[coluna].str.strip().replace(marcadores_ausentes, pd.NA)
    avaliacao[nome] = df
    ausencias_antes[nome] = df.isna().sum()
    numericas[nome] = [coluna for coluna in df if coluna not in ["date", "state", "time"]]
    print(nome)
    display(pd.DataFrame({"tipo_lido": df.dtypes.astype(str), "ausentes": df.isna().sum()}))
    print("Duplicatas idênticas:", original.duplicated().sum())
    print("Linhas com estado/data repetidos:", df.duplicated(["state", "date"], keep=False).sum())
    print("Categorias territoriais:", sorted(df["state"].dropna().unique()))

casos


,tipo_lido,ausentes
sno,string,0
date,string,0
time,string,0
state,string,0
confirmed_indian_national,string,17664
confirmed_foreign_national,string,17664
cured,string,0
deaths,string,0
confirmed,string,0


Duplicatas idênticas: 0
Linhas com estado/data repetidos: 0
Categorias territoriais: ['Andaman and Nicobar Islands', 'Andhra Pradesh', 'Arunachal Pradesh', 'Assam', 'Bihar', 'Bihar****', 'Cases being reassigned to states', 'Chandigarh', 'Chhattisgarh', 'Dadra and Nagar Haveli', 'Dadra and Nagar Haveli and Daman and Diu', 'Daman & Diu', 'Delhi', 'Goa', 'Gujarat', 'Haryana', 'Himachal Pradesh', 'Himanchal Pradesh', 'Jammu and Kashmir', 'Jharkhand', 'Karanataka', 'Karnataka', 'Kerala', 'Ladakh', 'Lakshadweep', 'Madhya Pradesh', 'Madhya Pradesh***', 'Maharashtra', 'Maharashtra***', 'Manipur', 'Meghalaya', 'Mizoram', 'Nagaland', 'Odisha', 'Puducherry', 'Punjab', 'Rajasthan', 'Sikkim', 'Tamil Nadu', 'Telangana', 'Telengana', 'Tripura', 'Unassigned', 'Uttar Pradesh', 'Uttarakhand', 'West Bengal']
testes


,tipo_lido,ausentes
date,string,0
state,string,0
total_samples,string,0
negative,string,9368
positive,string,10674


Duplicatas idênticas: 1
Linhas com estado/data repetidos: 2
Categorias territoriais: ['Andaman and Nicobar Islands', 'Andhra Pradesh', 'Arunachal Pradesh', 'Assam', 'Bihar', 'Chandigarh', 'Chhattisgarh', 'Dadra and Nagar Haveli and Daman and Diu', 'Delhi', 'Goa', 'Gujarat', 'Haryana', 'Himachal Pradesh', 'Jammu and Kashmir', 'Jharkhand', 'Karnataka', 'Kerala', 'Ladakh', 'Lakshadweep', 'Madhya Pradesh', 'Maharashtra', 'Manipur', 'Meghalaya', 'Mizoram', 'Nagaland', 'Odisha', 'Puducherry', 'Punjab', 'Rajasthan', 'Sikkim', 'Tamil Nadu', 'Telangana', 'Tripura', 'Uttar Pradesh', 'Uttarakhand', 'West Bengal']
vacinacao


,tipo_lido,ausentes
date,string,0
state,string,0
total_doses_administered,string,224
sessions,string,224
sites,string,224
first_dose_administered,string,224
second_dose_administered,string,224
male_doses_administered,string,384
female_doses_administered,string,384
transgender_doses_administered,string,384


Duplicatas idênticas: 0
Linhas com estado/data repetidos: 0
Categorias territoriais: ['Andaman and Nicobar Islands', 'Andhra Pradesh', 'Arunachal Pradesh', 'Assam', 'Bihar', 'Chandigarh', 'Chhattisgarh', 'Dadra and Nagar Haveli and Daman and Diu', 'Delhi', 'Goa', 'Gujarat', 'Haryana', 'Himachal Pradesh', 'India', 'Jammu and Kashmir', 'Jharkhand', 'Karnataka', 'Kerala', 'Ladakh', 'Lakshadweep', 'Madhya Pradesh', 'Maharashtra', 'Manipur', 'Meghalaya', 'Mizoram', 'Nagaland', 'Odisha', 'Puducherry', 'Punjab', 'Rajasthan', 'Sikkim', 'Tamil Nadu', 'Telangana', 'Tripura', 'Uttar Pradesh', 'Uttarakhand', 'West Bengal']


In [5]:
# Conferindo datas e contagens antes de alterar os dados.
diagnostico = []
for nome, df in avaliacao.items():
    datas = pd.to_datetime(df["date"], format=formatos_data[nome], errors="coerce")
    diagnostico.append({"base": nome, "campo": "date", "invalidos": int(datas.isna().sum())})
    for coluna in numericas[nome]:
        numeros = pd.to_numeric(df[coluna], errors="coerce")
        invalidos = df[coluna].notna() & (
            numeros.isna() | numeros.lt(0) | numeros.ge(2**63) | numeros.mod(1).ne(0)
        )
        diagnostico.append({"base": nome, "campo": coluna, "invalidos": int(invalidos.sum())})
display(pd.DataFrame(diagnostico))

,base,campo,invalidos
0,casos,date,0
1,casos,sno,0
2,casos,confirmed_indian_national,0
3,casos,confirmed_foreign_national,0
4,casos,cured,0
5,casos,deaths,0
6,casos,confirmed,0
7,testes,date,0
8,testes,total_samples,0
9,testes,negative,0


As contagens precisam ser inteiras e não negativas. Uma ausência é permitida e não será transformada em zero. Se surgirem valores inválidos, a preparação será interrompida para revisão.

Também verificaremos relações entre campos e quedas dos acumulados depois de ordenar as datas e corrigir as grafias. Esses alertas serão preservados: podem refletir revisões dos boletins. Valores máximos, por si só, não comprovam erros, pois os territórios têm tamanhos diferentes.

## 5. Limpeza e preparação dos dados

### Duplicatas, espaços e nomes territoriais

Removemos apenas linhas idênticas, sem considerar os metadados adicionados. Guardamos a posição e o nome territorial original para rastrear cada registro. Corrigimos somente as grafias já identificadas no projeto; territórios históricos continuam separados.

In [6]:
correcoes_estados = {
    "Karanataka": "Karnataka", "Himanchal Pradesh": "Himachal Pradesh",
    "Telengana": "Telangana", "Bihar****": "Bihar",
    "Madhya Pradesh***": "Madhya Pradesh", "Maharashtra***": "Maharashtra",
    "Daman & Diu": "Daman and Diu",
}
preparados = {}
duplicatas_removidas = {}
for nome, original in originais.items():
    repetidas = original.duplicated()
    duplicatas_removidas[nome] = int(repetidas.sum())
    if repetidas.any():
        print("Duplicatas removidas:", nome)
        display(original.loc[repetidas])
    df = original.rename(columns=mapa_colunas[nome]).copy()
    df.insert(0, "source_row", range(2, len(df) + 2))
    df["state_original"] = df["state"]
    df = df.loc[~repetidas].copy()
    for coluna in mapa_colunas[nome].values():
        df[coluna] = df[coluna].str.strip().replace(marcadores_ausentes, pd.NA)
    df["state"] = df["state"].replace(correcoes_estados)
    df["geographic_level"] = "state_or_union_territory"
    df.loc[df["state"].eq("India"), "geographic_level"] = "national"
    df.loc[df["state"].isin(["Unassigned", "Cases being reassigned to states"]), "geographic_level"] = "unassigned"
    preparados[nome] = df
    alterados = df["state_original"].ne(df["state"]).fillna(False)
    print(nome, "— registros com nome padronizado:", int(alterados.sum()))
    display(df.loc[alterados, ["state_original", "state"]].drop_duplicates())

casos — registros com nome padronizado: 433


,state_original,state
32,Telengana,Telangana
2890,Daman & Diu,Daman and Diu
15846,Bihar****,Bihar
17049,Madhya Pradesh***,Madhya Pradesh
17294,Himanchal Pradesh,Himachal Pradesh
17297,Karanataka,Karnataka
17338,Maharashtra***,Maharashtra


Duplicatas removidas: testes


,Date,State,TotalSamples,Negative,Positive
14863,2020-04-05,Uttar Pradesh,5255.0,4796,278.0


testes — registros com nome padronizado: 0


,state_original,state


vacinacao — registros com nome padronizado: 0


,state_original,state


### Datas, chaves e contagens

Casos e testes usam ano-mês-dia; vacinação usa dia/mês/ano. Após padronizar as chaves, conferimos se existe mais de um registro por território e data. Não escolhemos uma versão arbitrariamente quando houver conflito.

As contagens usam `Int64`, que aceita inteiros e ausências. A conversão remove apenas a parte decimal composta por zeros antes de ler o inteiro, evitando passar por números de ponto flutuante e perder precisão. Não há arredondamento.

In [7]:
for nome, df in preparados.items():
    datas = pd.to_datetime(df["date"], format=formatos_data[nome], errors="coerce")
    if datas.isna().any() or df["state"].isna().any():
        raise ValueError(f"Data ou território inválido em {nome}; revise antes de exportar.")
    df["date"] = datas.dt.strftime("%Y-%m-%d").astype("string")
    if df.duplicated(["state", "date"]).any():
        raise ValueError(f"Chaves repetidas após padronização em {nome}; revise antes de exportar.")

    for coluna in numericas[nome]:
        texto = df[coluna].str.replace(r"\.0+$", "", regex=True)
        if not texto.dropna().str.fullmatch(r"[0-9]+").all():
            raise ValueError(f"Contagem inválida em {nome}/{coluna}; revise antes de exportar.")
        try:
            # Converter texto diretamente para inteiro mantém a precisão.
            df[coluna] = texto.astype("Int64")
        except (ValueError, TypeError, OverflowError) as erro:
            raise ValueError(f"Contagem fora do intervalo Int64 em {nome}/{coluna}.") from erro
    preparados[nome] = df

In [8]:
# Padronizando a hora sem presumir um fuso horário.
horas_originais = preparados["casos"]["time"].str.upper()
horas = pd.Series(pd.NA, index=horas_originais.index, dtype="string")
for formato in ["%I:%M %p", "%H:%M", "%H:%M:%S"]:
    convertidas = pd.to_datetime(horas_originais, format=formato, errors="coerce")
    horas = horas.fillna(convertidas.dt.strftime("%H:%M:%S"))
if horas.isna().any():
    raise ValueError("Hora inválida em casos; revise antes de exportar.")
preparados["casos"]["time"] = horas

for nome, df in preparados.items():
    preparados[nome] = df.sort_values(["state", "date", "source_row"]).reset_index(drop=True)

### Alertas que serão mantidos

Com os tipos e nomes preparados, podemos comparar registros consecutivos de cada território. A comparação não atravessa valores ausentes. Não preenchemos lacunas nem corrigimos quedas dos acumulados.

Em casos, comparamos recuperados e óbitos com confirmados. Em vacinação, comparamos as doses. Não exigimos que positivos + negativos seja igual a amostras, porque as coberturas podem ser diferentes.

In [9]:
acumuladas = {
    "casos": ["confirmed", "cured", "deaths"],
    "testes": ["total_samples"],
    "vacinacao": ["total_doses_administered", "first_dose_administered", "second_dose_administered"],
}
quedas = []
for nome, df in preparados.items():
    for coluna in acumuladas[nome]:
        anterior = df.groupby("state")[coluna].shift()
        queda = df[coluna].lt(anterior).fillna(False)
        quedas.append({"base": nome, "campo": coluna, "quedas": int(queda.sum())})
display(pd.DataFrame(quedas))

casos = preparados["casos"]
vacinacao = preparados["vacinacao"]
inconsistencias = {
    "Recuperados > confirmados": casos["cured"] > casos["confirmed"],
    "Óbitos > confirmados": casos["deaths"] > casos["confirmed"],
    "Recuperados + óbitos > confirmados": casos["cured"] + casos["deaths"] > casos["confirmed"],
    "Primeira + segunda dose diferente do total": vacinacao["first_dose_administered"] + vacinacao["second_dose_administered"] != vacinacao["total_doses_administered"],
    "Segunda dose > primeira dose": vacinacao["second_dose_administered"] > vacinacao["first_dose_administered"],
}
display(pd.DataFrame([
    {"verificacao": regra, "ocorrencias": int(mascara.fillna(False).sum())}
    for regra, mascara in inconsistencias.items()
]))
for nome, df in preparados.items():
    print(nome, "— intervalos observados, sem excluir extremos")
    medidas = [coluna for coluna in numericas[nome] if coluna != "sno"]
    display(df[medidas].agg(["min", "max"]).T)

,base,campo,quedas
0,casos,confirmed,49
1,casos,cured,2
2,casos,deaths,1
3,testes,total_samples,23
4,vacinacao,total_doses_administered,57
5,vacinacao,first_dose_administered,64
6,vacinacao,second_dose_administered,3


,verificacao,ocorrencias
0,Recuperados > confirmados,0
1,Óbitos > confirmados,0
2,Recuperados + óbitos > confirmados,7
3,Primeira + segunda dose diferente do total,0
4,Segunda dose > primeira dose,0


casos — intervalos observados, sem excluir extremos


,min,max
confirmed_indian_national,0,177
confirmed_foreign_national,0,14
cured,0,6159676
deaths,0,134201
confirmed,0,6363442


testes — intervalos observados, sem excluir extremos


,min,max
total_samples,58,67897856
negative,0,83561026
positive,0,1638961


vacinacao — intervalos observados, sem excluir extremos


,min,max
total_doses_administered,7,513228400
sessions,0,35010311
sites,0,73933
first_dose_administered,7,400150406
second_dose_administered,0,113077994
male_doses_administered,0,270163622
female_doses_administered,2,239518609
transgender_doses_administered,0,98275
covaxin_doses_administered,0,62367416
covi_shield_doses_administered,7,446825051


## 6. Conjunto de dados resultante

As três bases continuam separadas, com todas as variáveis originais e três campos de rastreabilidade. Abaixo estão as dimensões, os períodos, os tipos finais e as ausências antes/depois. As verificações precisam passar antes de salvar qualquer CSV.

In [10]:
resumo = []
for nome, df in preparados.items():
    assert not df[["state", "date"]].isna().any().any(), f"Chave ausente em {nome}"
    assert not df.duplicated(["state", "date"]).any(), f"Chave repetida em {nome}"
    assert len(df) + duplicatas_removidas[nome] == len(originais[nome])
    assert df[numericas[nome]].ge(0).fillna(True).all().all()
    assert all(str(df[coluna].dtype) == "Int64" for coluna in numericas[nome])
    assert hashlib.sha256((pasta / arquivos[nome]).read_bytes()).hexdigest() == hashes_originais[nome]
    resumo.append({
        "base": nome, "entrada": len(originais[nome]), "saida": len(df),
        "duplicatas_removidas": duplicatas_removidas[nome], "colunas": len(df.columns),
        "inicio": df["date"].min(), "fim": df["date"].max(),
    })
    print(nome)
    display(pd.DataFrame({
        "tipo_final": df.dtypes.astype(str),
        "ausentes_antes": ausencias_antes[nome].reindex(df.columns).astype("Int64"),
        "ausentes_depois": df.isna().sum(),
    }))
    display(df.head(3))
display(pd.DataFrame(resumo))
print("Verificações finais aprovadas. Arquivos originais preservados.")

casos


,tipo_final,ausentes_antes,ausentes_depois
source_row,int64,<NA>,0
sno,Int64,0,0
date,string,0,0
time,string,0,0
state,string,0,0
confirmed_indian_national,Int64,17664,17664
confirmed_foreign_national,Int64,17664,17664
cured,Int64,0,0
deaths,Int64,0,0
confirmed,Int64,0,0


,source_row,sno,date,time,state,confirmed_indian_national,confirmed_foreign_national,cured,deaths,confirmed,state_original,geographic_level
0,367,366,2020-03-26,18:00:00,Andaman and Nicobar Islands,1,0,0,0,1,Andaman and Nicobar Islands,state_or_union_territory
1,394,393,2020-03-27,10:00:00,Andaman and Nicobar Islands,1,0,0,0,1,Andaman and Nicobar Islands,state_or_union_territory
2,422,421,2020-03-28,18:00:00,Andaman and Nicobar Islands,6,0,0,0,6,Andaman and Nicobar Islands,state_or_union_territory


testes


,tipo_final,ausentes_antes,ausentes_depois
source_row,int64,<NA>,0
date,string,0,0
state,string,0,0
total_samples,Int64,0,0
negative,Int64,9368,9368
positive,Int64,10674,10674
state_original,string,<NA>,0
geographic_level,str,<NA>,0


...

vacinacao


,tipo_final,ausentes_antes,ausentes_depois
source_row,int64,<NA>,0
date,string,0,0
state,string,0,0
total_doses_administered,Int64,224,224
sessions,Int64,224,224
sites,Int64,224,224
first_dose_administered,Int64,224,224
second_dose_administered,Int64,224,224
male_doses_administered,Int64,384,384
female_doses_administered,Int64,384,384


,source_row,date,state,total_doses_administered,sessions,sites,first_dose_administered,second_dose_administered,male_doses_administered,female_doses_administered,...,60_plus_years_doses_administered,18_44_years_individuals_vaccinated,45_60_years_individuals_vaccinated,60_plus_years_individuals_vaccinated,male_individuals_vaccinated,female_individuals_vaccinated,transgender_individuals_vaccinated,total_individuals_vaccinated,state_original,geographic_level
0,214,2021-01-16,Andaman and Nicobar Islands,23,2,2,23,0,12,11,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,23,Andaman and Nicobar Islands,state_or_union_territory
1,215,2021-01-17,Andaman and Nicobar Islands,23,2,2,23,0,12,11,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,23,Andaman and Nicobar Islands,state_or_union_territory
2,216,2021-01-18,Andaman and Nicobar Islands,42,9,2,42,0,29,13,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,42,Andaman and Nicobar Islands,state_or_union_territory


,base,entrada,saida,duplicatas_removidas,colunas,inicio,fim
0,casos,18110,18110,0,12,2020-01-30,2021-08-11
1,testes,16336,16335,1,8,2020-04-01,2021-08-10
2,vacinacao,7845,7845,0,27,2021-01-16,2021-08-16


Verificações finais aprovadas. Arquivos originais preservados.


O valor vazio em “ausentes antes” nos três campos adicionados indica que eles não existiam na fonte.

Os arquivos abaixo são substituídos a cada execução. Os originais nunca são sobrescritos. CSV não guarda tipos: ao reler, informe `Int64` para contagens e converta a data com o formato `%Y-%m-%d`.

No Colab, abra o painel **Arquivos**, entre em `data/processed` e baixe os três CSVs. Para guardar as saídas do notebook, use **Arquivo → Fazer download → Fazer download do .ipynb**.

In [9]:
destino = pasta / "data" / "processed"
destino.mkdir(parents=True, exist_ok=True)
for nome, df in preparados.items():
    arquivo_saida = destino / f"{nome}.csv"
    df.to_csv(arquivo_saida, index=False, encoding="utf-8", lineterminator="\n")
    print(f"Salvo: data/processed/{nome}.csv — {len(df)} registros")

Salvo: data/processed/casos.csv — 18110 registros

Salvo: data/processed/testes.csv — 16335 registros


Salvo: data/processed/vacinacao.csv — 7845 registros


### Limitações para as próximas etapas

- Ausência não significa zero. Algumas colunas têm muitas lacunas; selecione campos e períodos com preenchimento suficiente.
- Não some acumulados entre datas nem o total de India aos estados.
- Doses, indivíduos e amostras são medidas diferentes.
- Quedas e divergências foram preservadas, pois não há evidência para estimar correções.
- A consistência estrutural não comprova completude ou validade epidemiológica.

Não fizemos junções, interpolação, taxas populacionais ou inferências causais nesta RA1.